## Advanced_CI_w6
### CH10

### 10.1 지역 실험

In [2]:
import pandas as pd
import numpy as np

df = (pd.read_csv("online_mkt.csv")
      .astype({"date":"datetime64[ns]"})
      .query("post==0"))

df.head()

,app_download,population,city,state,date,post,treated
0,3066.0,12396372,sao_paulo,sao_paulo,2022-03-01,0,1
1,2701.0,12396372,sao_paulo,sao_paulo,2022-03-02,0,1
2,1927.0,12396372,sao_paulo,sao_paulo,2022-03-03,0,1
3,1451.0,12396372,sao_paulo,sao_paulo,2022-03-04,0,1
4,1248.0,12396372,sao_paulo,sao_paulo,2022-03-05,0,1


In [3]:
detectable_diff = df["app_download"].mean()*0.05
sigma_2 = df.groupby("city")["app_download"].mean().var()

float(np.ceil((sigma_2*16)/(detectable_diff)**2))

36663.0

In [4]:
df_piv = df.pivot(index = "date",columns = "city", values = "app_download")
df_piv.head()

city,ananindeua,aparecida_de_goiania,aracaju,belem,belford_roxo,belo_horizonte,brasilia,campinas,campo_grande,campos_dos_goytacazes,...,sao_bernardo_do_campo,sao_goncalo,sao_jose_dos_campos,sao_luis,sao_paulo,serra,sorocaba,teresina,uberlandia,vila_velha
date,,,,,,,,,,,,,,,,,,,,,
2022-03-01,11.0,54.0,65.0,49.0,52.0,194.0,449.0,326.0,305.0,42.0,...,208.0,148.0,173.0,28.0,3066.0,46.0,207.0,68.0,29.0,63.0
2022-03-02,5.0,20.0,42.0,44.0,25.0,130.0,367.0,172.0,261.0,24.0,...,185.0,157.0,137.0,4.0,2701.0,48.0,98.0,17.0,29.0,11.0
2022-03-03,2.0,0.0,0.0,24.0,13.0,112.0,219.0,292.0,171.0,28.0,...,96.0,72.0,171.0,6.0,1927.0,28.0,87.0,55.0,30.0,14.0
2022-03-04,0.0,0.0,11.0,4.0,34.0,56.0,265.0,154.0,175.0,43.0,...,123.0,102.0,0.0,5.0,1451.0,22.0,47.0,49.0,35.0,0.0
2022-03-05,5.0,5.0,0.0,16.0,12.0,45.0,135.0,122.0,189.0,35.0,...,99.0,53.0,112.0,12.0,1248.0,21.0,36.0,31.0,6.0,1.0


In [5]:
# 도시 가중치

f = (df.groupby("city")["population"].first()
    / df.groupby("city")["population"].first().sum())

In [6]:
y_avg = df_piv.dot(f)
geos = list(df_piv.columns)
n_tr = 5

In [7]:
np.random.seed(1)
rand_geos = np.random.choice(geos, n_tr, replace = False)
rand_geos

array(['manaus', 'recife', 'sao_bernardo_do_campo', 'salvador', 'aracaju'],
      dtype='<U23')

In [8]:
# CH9, Synthetic Control

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import (check_X_y, check_array, check_is_fitted)
import cvxpy as cp

class SyntheticControl(BaseEstimator, RegressorMixin):
    def __init__(self,fit_intercept=False):
        self.fit_intercept = fit_intercept

    def fit(self, y_pre_co, y_pre_tr):
        y_pre_co, y_pre_tr = check_X_y(y_pre_co, y_pre_tr)
        intercept = np.ones((y_pre_co.shape[0], 1))*self.fit_intercept
        X = np.concatenate([intercept, y_pre_co], axis = 1)
        w = cp.Variable(X.shape[1])
        
        objective = cp.Minimize(cp.sum_squares(X@w - y_pre_tr))
        constraints = [cp.sum(w[1:]) == 1, w[1:]>=0]
        
        problem = cp.Problem(objective, constraints)
        
        self.loss_ = problem.solve(verbose=False)
        self.intercept_ = float(w.value[0]) if self.fit_intercept else 0.0
        self.w_ = w.value[1:]

        self.is_fitted_ = True
        return self

    def predict(self, y_co):

        check_is_fitted(self)
        y_co = check_array(y_co)

        return y_co @ self.w_
    
    
        

In [9]:
def get_sc(geos, df_sc, y_mean_pre):
    model = SyntheticControl(fit_intercept = True)
    model.fit(df_sc[geos], y_mean_pre)

    selected_geos = geos[np.abs(model.w_) > 1e-5]

    return {"geos": selected_geos, "loss": model.loss_}

get_sc(rand_geos, df_piv, y_avg)

{'geos': array(['salvador', 'aracaju'], dtype='<U23'),
 'loss': 1598627.13369032}

In [10]:
def get_sc_st_combination(treatment_geos, df_sc, y_mean_pre):

    treatment_result = get_sc(treatment_geos, df_sc, y_mean_pre)

    remaining_geos = df_sc.drop(
        columns = treatment_result["geos"]
    ).columns

    control_result = get_sc(remaining_geos, df_sc, y_mean_pre)

    return {"st_geos": treatment_result["geos"],
            "sc_geos": control_result["geos"],
            "loss": treatment_result["loss"] + control_result["loss"]}
    
resulting_geos = get_sc_st_combination(rand_geos, df_piv, y_avg)

In [11]:
resulting_geos.get("st_geos")


array(['salvador', 'aracaju'], dtype='<U23')

In [12]:
len(resulting_geos.get("st_geos")) + len(resulting_geos.get("sc_geos"))

38

In [14]:
from joblib import Parallel, delayed
from toolz import partial

np.random.seed(1)
geo_samples = [np.random.choice(geos, n_tr, replace = False) for _ in range(1000)]

est_combination = partial(get_sc_st_combination, df_sc = df_piv, y_mean_pre = y_avg)

results = Parallel(n_jobs = 4)(delayed(est_combination)(geo_sample) for geo_sample in geo_samples)

In [15]:
resulting_geos = min(results, key = lambda x: x["loss"])
resulting_geos.get("st_geos")

array(['sao_bernardo_do_campo', 'joao_pessoa', 'feira_de_santana',
       'sao_paulo', 'porto_alegre'], dtype='<U23')

### 10.3 스위치백 실험


In [16]:
df = pd.read_csv("sb_exp_every.csv")
df.head()

,d,delivery_time,delivery_time_1,delivery_time_0,tau
0,1,2.84,2.84,5.84,-3.0
1,0,4.49,1.49,6.49,-5.0
2,0,7.27,2.27,8.27,-6.0
3,1,5.27,2.27,8.27,-6.0
4,1,5.59,4.59,10.59,-6.0


In [17]:
df_lags = df.assign(**{
    f"d_l{l}" : df["d"].shift(l) for l in range(7)
})

df_lags[[f"d_l{l}" for l in range(7)]].head()

,d_l0,d_l1,d_l2,d_l3,d_l4,d_l5,d_l6
0,1,NaN,NaN,NaN,NaN,NaN,NaN
1,0,1.0,NaN,NaN,NaN,NaN,NaN
2,0,0.0,1.0,NaN,NaN,NaN,NaN
3,1,0.0,0.0,1.0,NaN,NaN,NaN
4,1,1.0,0.0,0.0,1.0,NaN,NaN


In [19]:
import statsmodels.formula.api as smf
model = smf.ols("delivery_time ~" + "+".join([f"d_l{l}" for l in range(7)])
                ,data = df_lags).fit()
model.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,9.3270,0.461,20.246,0.000,8.414,10.240
d_l0,-2.9645,0.335,-8.843,0.000,-3.629,-2.300
d_l1,-1.8861,0.339,-5.560,0.000,-2.559,-1.213
d_l2,-1.0013,0.340,-2.943,0.004,-1.676,-0.327
d_l3,0.2594,0.341,0.762,0.448,-0.416,0.935
d_l4,0.1431,0.340,0.421,0.675,-0.531,0.817
d_l5,0.1388,0.340,0.408,0.684,-0.536,0.813
d_l6,0.5588,0.336,1.662,0.099,-0.108,1.225


In [21]:
# 절편 제거
tau_m_hat = model.params[1:].sum()
se_tau_m_hat = np.sqrt((model.bse[1:]**2).sum())
print("tau_m:", tau_m_hat)
print("95% CI:", [float(tau_m_hat - 1.96 * se_tau_m_hat), float(tau_m_hat + 1.96 * se_tau_m_hat)])

tau_m: -4.751686115272039
95% CI: [-6.508718378154574, -2.994653852389504]


In [22]:
# 시차 0, 1, 2
tau_m_hat = model.params[1:4].sum()
se_tau_m_hat = np.sqrt((model.bse[1:4]**2).sum())
print("tau_m:", tau_m_hat)
print("95% CI:", [float(tau_m_hat - 1.96 * se_tau_m_hat), float(tau_m_hat + 1.96 * se_tau_m_hat)])

tau_m: -5.851856895442294
95% CI: [-7.000105171362165, -4.703608619522424]


In [23]:
rad_points_3 = np.array([True, False, False]*(2))
rad_points_3

array([ True, False, False,  True, False, False])

In [24]:
rad_points_3.cumsum()

array([1, 1, 1, 2, 2, 2])

In [25]:
from numpy.lib.stride_tricks import sliding_window_view
m=2
sliding_window_view(rad_points_3.cumsum(), window_shape = m+1)

array([[1, 1, 1],
       [1, 1, 2],
       [1, 2, 2],
       [2, 2, 2]])

In [ ]:
np.diff(sliding_window_view(rad_points_3.cumsum(), 3), axis = 1)

array([[0, 0],
       [0, 1],
       [1, 0],
       [0, 0]])

In [31]:
n_rand_windows = np.concatenate([
    [np.nan]*m,
    np.diff(sliding_window_view(rad_points_3.cumsum(), 3),
            axis=1).sum(axis=1)+1
])

n_rand_windows

array([nan, nan,  1.,  2.,  2.,  1.])

In [32]:
p = 0.5
p**n_rand_windows

array([ nan,  nan, 0.5 , 0.25, 0.25, 0.5 ])

In [33]:
def compute_p(rand_points, m, p=0.5):
    n_windows_last_m = np.concatenate([
        [np.nan]*m,
        np.diff(sliding_window_view(rand_points.cumsum(), m+1),
                axis=1).sum(axis=1)+1
    ])
    return p**n_windows_last_m

compute_p(np.ones(6)==1, 2, 0.5)

array([  nan,   nan, 0.125, 0.125, 0.125, 0.125])

In [34]:
rand_points = np.array([True, False, False, True, False, True, False])
compute_p(rand_points, 2, 0.5)

array([ nan,  nan, 0.5 , 0.25, 0.25, 0.25, 0.25])

In [35]:
def last_m_d_equal(d_vec, d, m):
    return np.concatenate([
        [np.nan]*m,
        np.all(sliding_window_view(d_vec, m+1) == d, axis = 1)
    ])

print(last_m_d_equal([1,1,1,0,0,0], 1, m=2))
print(last_m_d_equal([1,1,1,0,0,0], 0, m=2))

[nan nan  1.  0.  0.  0.]
[nan nan  0.  0.  0.  1.]


In [36]:
def ipw_switchback(d, y, rand_points, m, p=0.5):
    p_last_m_equal_1 = compute_p(rand_points, m, p)
    p_last_m_equal_0 = compute_p(rand_points, m, 1-p)

    last_m_is_1 = last_m_d_equal(d, 1, m)
    last_m_is_0 = last_m_d_equal(d, 0, m)

    y1_rec = y*last_m_is_1/p_last_m_equal_1
    y0_rec = y*last_m_is_0/p_last_m_equal_0

    return np.mean((y1_rec - y0_rec)[m:])

In [38]:
float(ipw_switchback(df["d"], df["delivery_time"], np.ones(len(df))==1, m=2, p=0.5))

-7.426440677966101

In [40]:
m = 2
T = 12
n = T/m
np.isin(
    np.arange(1,T+1),
    [1] + [i*m+1 for i in range(2, int(n)-1)]
)*1

array([1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0])

In [41]:
m = 3
T = 15
n = T/m
np.isin(
    np.arange(1,T+1),
    [1] + [i*m+1 for i in range(2, int(n)-1)]
)*1

array([1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0])

In [42]:
df_opt = pd.read_csv("sb_exp_opt.csv")
df_opt.head()

,rand_points,d,delivery_time
0,True,0,5.84
1,False,0,5.40
2,False,0,8.86
3,False,0,8.79
4,True,0,10.93


In [44]:
tau_hat = ipw_switchback(df_opt["d"],
                        df_opt["delivery_time"],
                        df_opt["rand_points"],
                        m=2, p=0.5)
float(tau_hat)


-9.921016949152545

In [45]:
np.vstack(np.hsplit(np.array([1,1,1,2,2,3]), 3))

array([[1, 1],
       [1, 2],
       [2, 3]])

In [46]:
np.diff(np.vstack(np.hsplit(np.array([1,1,0,0,0,0]), 3))[:, 0]) == 0


array([False,  True])

In [47]:
def var_opt_design(d_opt, y_opt, T, m):
    assert ((T/m == T/m)
            & (T/m >=4)), "T must be divisible by m and T/m >= 4"
    # 1st block 버리기
    y_m_blocks = np.vstack(np.hsplit(y_opt, int(T/m))).sum(axis=1)[1:]
    # 1st column value
    d_m_blocks = np.vstack(np.hsplit(d_opt, int(T/m))[1:])[:,0]

    return (
        8*y_m_blocks[0]**2 + (32*y_m_blocks[1:-1]**2*(np.diff(d_m_blocks)==0)[:-1]).sum() + 8*y_m_blocks[-1]**2
    ) / (T-m)**2

In [50]:
se_hat = np.sqrt(var_opt_design(df_opt["d"],
                                df_opt["delivery_time"],
                                T=120, m=2))
[float(tau_hat - 1.96*se_hat), float(tau_hat + 1.96*se_hat)]

c:\Users\User\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


[-18.490627362048095, -1.351406536256997]